# Venusian PCW Detector · 科学图表

用原始 1 Hz MAG 和事件清单生成图表。太阳黑子观测与弓激波模型参数随仓库提供。全任务连续搜索耗时较长，通过参数单独开启。每组图保存为 PNG 和 PDF，输出在 `build/figures/`。

In [ ]:
# 本机原始数据目录。None 表示沿用 VENUS_EXPRESS_RAW_DATA_DIR。
MAG_DIR = None
EVENTS_CSV = None  # None 使用附带的事件清单
EVENT_DAYS = None  # 例如 ["2006-04-24"] 做单日诊断
WORKERS = 4
DEVICE = "cpu"

RUN_CATALOGUE = True
RUN_MISSION_SEARCH = False
SEARCH_DAYS = None  # 开启搜索时，None 使用本机数据目录的全部观测日
DRAW_MAG_EXAMPLES = True
SHOW_INLINE = True

FONT_FAMILIES = []
FONT_PATHS = []

## 准备环境

In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display
from venus_express.repro.settings import BUILD, FIGURES, display_path, read

os.environ.setdefault("MPLCONFIGDIR", str(BUILD / "matplotlib"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
if MAG_DIR is not None:
    os.environ["VENUS_EXPRESS_RAW_DATA_DIR"] = str(Path(MAG_DIR).expanduser().resolve())

from venus_express.repro.raw import observation_days
from venusian_pcw import figures
from venusian_pcw.fonts import plot_families
from venusian_pcw.notebook import day_flags, run_cli

font_stack = plot_families(FONT_FAMILIES, FONT_PATHS)
has_mag = bool(os.environ.get("VENUS_EXPRESS_RAW_DATA_DIR"))
if not has_mag:
    print("设置 MAG_DIR 后可运行事件测量、搜索和原始观测示例。")

generated = []
def draw_figure(name, render, *args):
    with figures.figure_style(font_stack):
        render(*args)
    path = FIGURES / f"{name}.png"
    generated.append(path)
    if SHOW_INLINE:
        display(Image(filename=str(path), width=850))

## 太阳活动与参考弓激波

黑子图读取 SILSO 月均观测。弓激波图按参数计算两条模型曲线。

In [ ]:
draw_figure("07_solar_cycle", figures.solar_cycle)
draw_figure("09_shock_geometry", figures.shock_models)

## 事件空间分布和氢密度

从 BAM 逐事件测量；样本满足拟合要求时生成氢密度图。

In [ ]:
catalogue_dir = None
has_annual_energy = False
if RUN_CATALOGUE and has_mag:
    event_flags = ["--events", Path(EVENTS_CSV).expanduser().resolve()] if EVENTS_CSV else []
    run_cli("catalogue", *event_flags, *day_flags(EVENT_DAYS), "--workers", WORKERS)
    catalogue_dir = BUILD / "raw_catalogue"
    if EVENT_DAYS:
        catalogue_dir /= "diagnostic"
    measured = read(catalogue_dir / "summary.json")
    if measured["energy_available"]:
        draw_figure("12_rx_manual_energy", figures.catalogue_map, catalogue_dir)
    if "density" in measured:
        draw_figure("06_h_density", figures.density, catalogue_dir)
    has_annual_energy = "wave_energy" in measured

## 连续检出、空间曝光与年际变化

开启搜索时，由权重和 BAM 重算候选、逐日曝光及空间发生率。

In [ ]:
has_occurrence = RUN_MISSION_SEARCH and has_mag
if has_occurrence:
    run_cli("search", "mission", *day_flags(SEARCH_DAYS), "--workers", WORKERS, "--device", DEVICE)
    run_cli("replay", "--workers", WORKERS)
    draw_figure("11_rx_cluster_occurrence", figures.cluster_map)
if has_occurrence or has_annual_energy:
    draw_figure("05_annual_response", figures.annual_response,
                catalogue_dir if has_annual_energy else None, has_occurrence)

## 原始 MAG 观测示例

In [ ]:
if DRAW_MAG_EXAMPLES and has_mag:
    dates = set(observation_days())
    if "2014-02-22" in dates:
        draw_figure("04_pcw_event", figures.wave_event)
    if "2008-12-03" in dates:
        draw_figure("10_radial_imf_event", figures.radial_event)
    if {"2011-02-20", "2014-02-22", "2009-10-12"} <= dates:
        draw_figure("13_rx_case_orbits", figures.rx_examples)

print(f"本次生成 {len(generated)} 张图。目录：{display_path(FIGURES)}")